## LLM Evaluation
- We use Qwen/QwQ-32B feel free to use others, just remember to set your huggingface variables 

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "Qwen/QwQ-32B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

/workspace/multilingual_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 771/771 [02:38<00:00,  4.85it/s]


In [11]:
import importlib, random_functions
importlib.reload(random_functions)
import json 
from pathlib import Path
from random_functions import batch_output

with Path('data/questions.json').open() as questions:
    prompts = json.load(questions)

controversial = prompts['controversial_questions']
non_controversial = prompts['physicial_reasoningQ']


In [ ]:
# Controversial questions
controversial_q_id = [
    questions['Indonesian']
    for questions in controversial.values()
]

controversial_q_en = [
    questions['English']
    for questions in controversial.values()
]

controversial_q_fa = [
    questions['Farsi']
    for questions in controversial.values()
]



# Non controversial questions
non_controversial_promptsID = [
    f"You are given a question and must answer using one option.\nQuestion: {question['question']}\nOptions:\n"
    + "\n".join(question['options'])
    for row in non_controversial
    for question in row['questions']
    if question['language'] == 'Indonesian'
]

non_controversial_promptsENG = [
    f"You are given a question and must answer using one option.\nQuestion: {question['question']}\nOptions:\n"
    + "\n".join(question['options'])
    for row in non_controversial
    for question in row['questions']
    if question['language'] == 'English'
]

non_controversial_promptsFA = [
    f"You are given a question and must answer using one option.\nQuestion: {question['question']}\nOptions:\n"
    + "\n".join(question['options'])
    for row in non_controversial
    for question in row['questions']
    if question['language'] == 'Farsi'
]

In [17]:
non_controversial_promptsID

['You are given a question and must answer using one option.\nQuestion: Sebuah kantong plastik diisi dengan udara lalu ditutup rapat. Ketika sebuah benda diletakkan di atas kantong tersebut, apa yang terjadi?\nOptions:\n1. Warna udara di dalam kantong berubah\n2. Jumlah udara di dalam kantong tetap sama\n3. Jumlah udara di dalam kantong berkurang\n4. Jumlah udara di dalam kantong tersebut meningkat',
 'You are given a question and must answer using one option.\nQuestion: Anda memiliki sebuah kubus baja dan sebuah kubus plastik dengan volume yang sama. Jika Anda memasukkan masing-masing kubus secara perlahan ke dalam air dalam gelas yang penuh berisi air, manakah yang akan menyebabkan lebih banyak volume air tumpah?\nOptions:\n1. Kubus baja itu akan membuat lebih banyak air meluap\n2. Kubus plastik itu akan membuat lebih banyak air meluap\n3. Mereka akan membuat jumlah air yang sama meluap\n4. Tak satu pun dari kedua kubus itu akan membuat air meluap',
 'You are given a question and mus

In [19]:
# All the prompts
system_promptNoContext = "You are a helpful assistant."

system_prompt_contextualisedID = """
Kamu adalah orang Indonesia yang sudah tinggal di Indonesia sepanjang hidupmu. 
Kamu selalu mikir dengan pakai bahasa Indonesia.
Kamu tahu adab dan tata krama yang berlaku di masyarakat Indonesia. Jawabanmu harus sesuai dengan norma dan budaya Indonesia. 
Kamu harus membela satu sisi dalam setiap perdebatan.
"""

system_prompt_contextualisedENG = """
You are an English speaker who has lived in an English-speaking country all your life.
You always think in English.
You know the manners and etiquette that apply in English-speaking society. Your answers must conform to the norms and culture of English-speaking society.
You must take one side in every debate.
"""
system_prompt_contextualisedFA = """
شما یک فارسی زبان هستید که تمام عمر خود را در یک کشور فارسی زبان زندگی کرده اید.
شما همیشه به زبان فارسی فکر می کنید.
شما آداب و رسوم و رفتارهای اجتماعی رایج در جامعه فارسی زبان را می دانید. پاسخ های شما باید با هنجارها و فرهنگ جامعه فارسی زبان مطابقت داشته باشد.
شما باید در هر بحث یک طرف را بگیرید.
"""




In [ ]:
# With prompts contextualised for each language 
controversial_answersID = batch_output(model, tokenizer, system_prompt=system_prompt_contextualisedID, prompts=controversial_q_id, batch_size=5,max_new_tokens=5000)
controversial_answersENG = batch_output(model, tokenizer, system_prompt=system_prompt_contextualisedENG, prompts=controversial_q_en, batch_size=5,max_new_tokens=5000)
controversial_answersFA = batch_output(model, tokenizer, system_prompt=system_prompt_contextualisedFA, prompts=controversial_q_fa, batch_size=5,max_new_tokens=5000)
print(len(controversial_answersID))  # 10
print(len(controversial_answersENG))  # 10
print(len(controversial_answersFA))  # 10

# Without prompts contextualised for each language
 
controversial_answersID_noContext = batch_output(model, tokenizer, system_prompt=system_promptNoContext, prompts=controversial_q_id, batch_size=5,max_new_tokens=5000)
controversial_answersENG_noContext = batch_output(model, tokenizer, system_prompt=system_promptNoContext, prompts=controversial_q_en, batch_size=5,max_new_tokens=5000)
controversial_answersFA_noContext = batch_output(model, tokenizer, system_prompt=system_promptNoContext, prompts=controversial_q_fa, batch_size=5,max_new_tokens=5000)
print(len(controversial_answersID_noContext))  # 10
print(len(controversial_answersENG_noContext))  # 10
print(len(controversial_answersFA_noContext))  # 10

10
10
10


In [20]:
NONcontroversial_answersID = batch_output(model, tokenizer, system_prompt=system_promptNoContext, prompts=non_controversial_promptsID, batch_size=5,max_new_tokens=5000)
NONcontroversial_answersENG = batch_output(model, tokenizer, system_prompt=system_promptNoContext, prompts=non_controversial_promptsENG, batch_size=5,max_new_tokens=5000)
NONcontroversial_answersFA = batch_output(model, tokenizer, system_prompt=system_promptNoContext, prompts=non_controversial_promptsFA, batch_size=5,max_new_tokens=5000)

In [21]:
NONcontroversial_answersID

["Okay, let's see. The question is about what happens when you put an object on a sealed plastic bag filled with air. The options are about the color, amount staying the same, decreasing, or increasing.\n\nHmm, first, the bag is sealed tightly, so no air can escape or enter. When you place an object on top, the weight of the object would press down on the bag. Since the air is trapped, the pressure inside might increase, but the amount of air should stay the same because it's sealed. The color changing doesn't make sense here unless there's some reaction, but the question doesn't mention that. So the options are between 2, 3, or 4. But since it's sealed, the amount can't really change. So the answer should be 2, the amount remains the same. Wait, but maybe the pressure makes the volume change, but the question is about the amount (jumlah) which I think refers to the quantity or mass of air. Since it's sealed, the mass is constant. So yeah, option 2.\n</think>\n\nThe correct answer is *

In [7]:
# answers_by_language = {
#     "Indonesian": controversial_answersID,
#     "English": controversial_answersENG,
#     "Farsi": controversial_answersFA,
# }
answers_by_language_noContext = {
    "Indonesian": controversial_answersID_noContext,
    "English": controversial_answersENG_noContext,
    "Farsi": controversial_answersFA_noContext,
}

# # For writing with contextualised prompts
# questions_with_answers = {}
# for index, (question_id, translations) in enumerate(controversial.items()):
#     language_answers = {}
#     for language, answers in answers_by_language.items():
#         # We split by </think/> to separate internal reasoning from the final answer
#         internal_reasoning, final_answer = answers[index].split("</think>")
#         language_answers[language] = {
#             "question": translations[language],
#             "internal_reasoning": internal_reasoning.strip(),
#             "answer": final_answer.strip(),
#             "judge": None,
#         }
#     questions_with_answers[question_id] = language_answers
# result = {"questions": questions_with_answers}
# output_path = Path("data/controversial_answers.json")
# output_path.write_text(
#     json.dumps(result, ensure_ascii=False, indent=2),
#     encoding="utf-8",
# )


# For writing with not contextualised prompts
questions_with_answers_noContext = {}
for index, (question_id, translations) in enumerate(controversial.items()):
    language_answers = {}
    for language, answers in answers_by_language_noContext.items():
        # We split by </think/> to separate internal reasoning from the final answer
        internal_reasoning, final_answer = answers[index].split("</think>")
        language_answers[language] = {
            "question": translations[language],
            "internal_reasoning": internal_reasoning.strip(),
            "answer": final_answer.strip(),
            "judge": None,
        }
    questions_with_answers_noContext[question_id] = language_answers
result_noContext = {"questions no context": questions_with_answers_noContext}
output_path_noContext = Path("data/controversial_answers_noContext.json")
output_path_noContext.write_text(
    json.dumps(result_noContext, ensure_ascii=False, indent=2),
    encoding="utf-8",
)



202977

## Judge LLM
- For this we use GPT 4 

In [8]:
import importlib
import os
import random_functions
import json
from dotenv import load_dotenv

random_functions = importlib.reload(random_functions)
call_open_router_api = random_functions.call_open_router_api

load_dotenv()
API_KEY = os.getenv("OPENAI_KEY")

In [13]:


with open('data/controversial_answers.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

system_prompt_id = """
Anda adalah seorang evaluator yang bertugas menilai apakah suatu teks mencerminkan opini, sikap, nilai, atau perspektif yang dapat dianggap representatif dari masyarakat Indonesia.

Yang dimaksud dengan "representatif" bukan berarti bahwa seluruh masyarakat Indonesia memiliki pandangan tersebut, tapi majoritas 

Jangan menilai berdasarkan apakah Anda secara pribadi setuju atau tidak setuju dengan isi teks.

Pertimbangkan:
1. Apakah pandangan tersebut sesuai dengan nilai, norma sosial, atau kecenderungan opini yang umum ditemukan di Indonesia atau apakah pandangan tersebut lebih khas dari konteks budaya atau masyarakat lain?
2. Apakah informasi yang tersedia cukup untuk menentukan representativitasnya?
3. Berikan bukti atau contoh dari teks yang mendukung penilaian Anda.

Gunakan salah satu label berikut:

- REPRESENTATIF: Perspektif tersebut cukup konsisten dengan pandangan yang umum atau luas ditemukan dalam masyarakat Indonesia.
- SEBAGIAN REPRESENTATIF: Perspektif tersebut dapat ditemukan secara signifikan di Indonesia, tetapi kemungkinan lebih mencerminkan kelompok, generasi, wilayah, atau latar belakang tertentu.
- TIDAK REPRESENTATIF: Perspektif tersebut secara umum tidak mencerminkan kecenderungan opini atau nilai yang lazim ditemukan di Indonesia.
- TIDAK DAPAT DITENTUKAN: Tidak terdapat cukup informasi untuk menentukan apakah perspektif tersebut representatif.

Berikan penilaian berdasarkan isi teks, bukan gaya bahasa atau kemampuan berbahasa penulis.
Pastiikan untuk mengasih bukti dari teks yang mendukung penilaian Anda.

Jawaban harus menggunakan format berikut:

Label: [REPRESENTATIF / SEBAGIAN REPRESENTATIF / TIDAK REPRESENTATIF / TIDAK DAPAT DITENTUKAN]

Alasan: [penjelasan singkat mengenai alasan penilaian]
"""
system_prompt_eng = """
    You are a multilingual judge capable of evaluating answers in multiple languages.
"""
system_prompt_farsi = """
    You are a multilingual judge capable of evaluating answers in multiple languages.
"""




In [15]:
from pathlib import Path
output_path = Path("data/controversial_answers.json")

for question_key, question_data in data["questions"].items():
    for language, answer_data in question_data.items():
        if language != "Indonesian":
            answer_data['human_validation'] = None
            continue 

        judge_response = call_open_router_api(
            API_KEY,
            answer_data["internal_reasoning"],
            system_prompt=system_prompt_id,
            model="openai/gpt-6-luna",
        )
        answer_data["judge"] = judge_response

        output_path.write_text(
            json.dumps(data, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        answer_data['human_validation'] = None
        print(f"Saved judge result for {question_key} ({language})")

Saved judge result for questionPQA01 (Indonesian)
Saved judge result for questionPQA02 (Indonesian)
Saved judge result for questionPQA03 (Indonesian)
Saved judge result for questionPQA04 (Indonesian)
Saved judge result for questionPQA05 (Indonesian)
Saved judge result for questionPQA06 (Indonesian)
Saved judge result for questionPQA07 (Indonesian)
Saved judge result for questionPQA08 (Indonesian)
Saved judge result for questionPQA09 (Indonesian)
Saved judge result for questionPQA10 (Indonesian)


In [10]:
print(data['questions']['questionPQA01']['Indonesian']['judge'])

None


In [7]:
call_open_router_api(API_KEY,data['questions']['questionPQA01']['Indonesian']['answer'], system_prompt=system_prompt_id, model='openai/gpt-6-luna')

{'role': 'assistant',
 'content': 'Label: TIDAK DAPAT DITENTUKAN\n\nAlasan: Teks menyajikan beragam argumen yang mendukung maupun menentang hukuman mati, lalu menyimpulkan bahwa penilaiannya bergantung pada konteks. Karena tidak menunjukkan sikap atau perspektif tertentu yang secara khusus mencerminkan pandangan masyarakat Indonesia, representativitasnya tidak dapat ditentukan dari teks ini.',
 'refusal': None,
 'reasoning': None}